In [ ]:
# Auto-Fix Working Directory 
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory set to:", os.getcwd())


# 🧪 Lesson 15 — ADMET Prediction for Top Hits

> **Author:** Omar Hamouda   
> **Environment:** WSL/Ubuntu | VS Code | Conda (biochem)  
> **Status:** ✅ Starting  
> **Purpose:** Evaluate drug-likeness of top docking candidates

---

## 📌 Overview

After docking (Lesson 14/14b), we identified **607, BAH, 13U, 12U** as top hits.
But can they actually become drugs?

**ADMET answers 5 critical questions:**

| Letter | Meaning | Question |
|--------|---------|----------|
| **A** | Absorption | Can it be absorbed orally? |
| **D** | Distribution | Does it reach the target? |
| **M** | Metabolism | How long does it last? |
| **E** | Excretion | How is it removed? |
| **T** | Toxicity | Is it safe? |

---

## 🎯 The Pipeline

1.Load top hits (SDF files)


2.Calculate physicochemical properties (RDKit)

3.Check Lipinski's Rule of 5

4.Check Veber's Rules

5.Visualize comparison (Radar chart)

5.Save results

---

## 📁 Required Files

- `docking/results/fixed/607_fixed.sdf`
- `docking/results/fixed/BAH_fixed.sdf`
- `docking/results/fixed/13U_fixed.sdf`
- `docking/results/fixed/12U_fixed.sdf`

---

## 🧬 Lipinski's Rule of 5

A molecule is likely **orally bioavailable** if:

| Property | Rule |
|----------|------|
| **Molecular Weight** | ≤ 500 Da |
| **LogP** | ≤ 5 |
| **H-Bond Donors** | ≤ 5 |
| **H-Bond Acceptors** | ≤ 10 |

**Verdict:**
- **0 violations** → Excellent
- **1 violation** → Acceptable  
- **2+ violations** → Poor drug-likeness

---

## 🧬 Veber's Rules

Additional criteria for oral drugs:

| Property | Rule |
|----------|------|
| **Rotatable Bonds** | ≤ 10 |
| **TPSA** | ≤ 140 Ų |

---

**Let's start!** 🚀

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 2: Setup & Imports
# ═══════════════════════════════════════════════════════════

import os
import warnings
warnings.filterwarnings("ignore")

# Data manipulation
import pandas as pd
import numpy as np

# Cheminformatics
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski, rdMolDescriptors

# PAINS filter
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

# Visualization
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import matplotlib.cm as cm

# ─── Report versions ───
print("=" * 60)
print("  Environment Check")
print("=" * 60)
print(f"\n✅ RDKit:      {Chem.rdBase.rdkitVersion}")
print(f"✅ pandas:     {pd.__version__}")
print(f"✅ numpy:      {np.__version__}")
print("\n" + "=" * 60)
print("  🎉 All tools ready!")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 3: Load Top Hit Ligands from Lesson 14b
# ═══════════════════════════════════════════════════════════

# ─── Define paths ───
FIXED_DIR = "docking/results/fixed"
TOP_HITS = ["607", "BAH", "13U", "12U"]

# ─── Load each ligand ───
ligands = {}

print("=" * 60)
print("  Loading Top Hit Ligands")
print("=" * 60)

for lid in TOP_HITS:
    sdf_path = f"{FIXED_DIR}/{lid}_fixed.sdf"
    
    if not os.path.exists(sdf_path):
        print(f"  ❌ {lid:5s} — file not found")
        continue
    
    mol = Chem.MolFromMolFile(sdf_path, removeHs=False)
    
    if mol is None:
        print(f"  ❌ {lid:5s} — cannot parse SDF")
        continue
    
    ligands[lid] = mol
    n_atoms = mol.GetNumAtoms()
    n_heavy = mol.GetNumHeavyAtoms()
    print(f"  ✅ {lid:5s} — {n_atoms} atoms ({n_heavy} heavy)")

print("\n" + "=" * 60)
print(f"  Loaded: {len(ligands)} / {len(TOP_HITS)} ligands")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 4: Compute Molecular Descriptors
# ═══════════════════════════════════════════════════════════

def compute_descriptors(mol):
    """Compute all ADMET-relevant descriptors for a molecule."""
    return {
        "MW":             round(Descriptors.MolWt(mol), 2),
        "LogP":           round(Descriptors.MolLogP(mol), 2),
        "HBD":            Lipinski.NumHDonors(mol),
        "HBA":            Lipinski.NumHAcceptors(mol),
        "TPSA":           round(Descriptors.TPSA(mol), 2),
        "RotB":           Lipinski.NumRotatableBonds(mol),
        "Rings":          Lipinski.RingCount(mol),
        "Aromatic_Rings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "Heavy_Atoms":    mol.GetNumHeavyAtoms(),
        "Fraction_Csp3":  round(rdMolDescriptors.CalcFractionCSP3(mol), 3),
        "Mol_Refract":    round(Descriptors.MolMR(mol), 2),
        "Formal_Charge":  Chem.GetFormalCharge(mol),
    }


# ─── Compute for each ligand ───
print("=" * 70)
print("  Molecular Descriptors")
print("=" * 70)

descriptors_data = {}

for lid, mol in ligands.items():
    props = compute_descriptors(mol)
    descriptors_data[lid] = props
    print(f"\n▶ {lid}:")
    for key, value in props.items():
        print(f"   {key:18s} = {value}")

# ─── Build DataFrame ───
df_descriptors = pd.DataFrame(descriptors_data).T
df_descriptors.index.name = "Ligand"

print("\n" + "=" * 70)
print("  Summary Table")
print("=" * 70)
print(df_descriptors.to_string())
print("=" * 70)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 5: Lipinski's Rule of 5
# ═══════════════════════════════════════════════════════════

# ─── Lipinski's thresholds ───
LIPINSKI_RULES = {
    "MW":   {"threshold": 500, "direction": "<=", "name": "Molecular Weight"},
    "LogP": {"threshold": 5,   "direction": "<=", "name": "Lipophilicity"},
    "HBD":  {"threshold": 5,   "direction": "<=", "name": "H-Bond Donors"},
    "HBA":  {"threshold": 10,  "direction": "<=", "name": "H-Bond Acceptors"},
}

def check_lipinski(props):
    """Check Lipinski's Rule of 5 for a molecule."""
    results = {}
    violations = 0
    
    for key, rule in LIPINSKI_RULES.items():
        value = props[key]
        threshold = rule["threshold"]
        passed = value <= threshold
        results[f"{key}_pass"] = passed
        if not passed:
            violations += 1
    
    results["violations"] = violations
    
    if violations == 0:
        results["verdict"] = "Excellent"
    elif violations == 1:
        results["verdict"] = "Acceptable"
    else:
        results["verdict"] = "Poor"
    
    return results


# ─── Apply to each ligand ───
print("=" * 75)
print("  Lipinski's Rule of 5 — Analysis")
print("=" * 75)

lipinski_data = {}

for lid, props in descriptors_data.items():
    lip = check_lipinski(props)
    lipinski_data[lid] = lip
    
    print(f"\n▶ {lid}:")
    for key, rule in LIPINSKI_RULES.items():
        value = props[key]
        threshold = rule["threshold"]
        icon = "✅" if lip[f"{key}_pass"] else "❌"
        print(f"   {icon} {rule['name']:20s} = {value:8} (limit: {threshold})")
    
    print(f"\n   Violations: {lip['violations']}")
    print(f"   Verdict:    {lip['verdict']}")

# ─── Summary Table ───
print("\n" + "=" * 75)
print("  Lipinski Summary Table")
print("=" * 75)

df_lipinski = pd.DataFrame(lipinski_data).T
df_lipinski = df_lipinski[["violations", "verdict"]]
print(df_lipinski.to_string())
print("=" * 75)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 6: Veber's Rules — Oral Bioavailability
# ═══════════════════════════════════════════════════════════

# ─── Veber's thresholds ───
VEBER_RULES = {
    "RotB": {"threshold": 10,  "name": "Rotatable Bonds"},
    "TPSA": {"threshold": 140, "name": "Topological Polar Surface Area"},
}

def check_veber(props):
    """Check Veber's Rules for oral bioavailability."""
    results = {}
    violations = 0
    
    for key, rule in VEBER_RULES.items():
        value = props[key]
        threshold = rule["threshold"]
        passed = value <= threshold
        results[f"{key}_pass"] = passed
        if not passed:
            violations += 1
    
    results["violations"] = violations
    results["verdict"] = "Pass" if violations == 0 else "Fail"
    
    return results


# ─── Apply to each ligand ───
print("=" * 75)
print("  Veber's Rules — Analysis")
print("=" * 75)

veber_data = {}

for lid, props in descriptors_data.items():
    veb = check_veber(props)
    veber_data[lid] = veb
    
    print(f"\n▶ {lid}:")
    for key, rule in VEBER_RULES.items():
        value = props[key]
        threshold = rule["threshold"]
        icon = "✅" if veb[f"{key}_pass"] else "❌"
        print(f"   {icon} {rule['name']:35s} = {value:8} (limit: {threshold})")
    
    print(f"\n   Violations: {veb['violations']}")
    print(f"   Verdict:    {veb['verdict']}")

# ─── Summary Table ───
print("\n" + "=" * 75)
print("  Veber Summary Table")
print("=" * 75)

df_veber = pd.DataFrame(veber_data).T
df_veber = df_veber[["violations", "verdict"]]
print(df_veber.to_string())
print("=" * 75)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 7: PAINS Filter — Detect Problematic Compounds
# ═══════════════════════════════════════════════════════════

# ─── Setup PAINS filter catalog ───
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
pains_catalog = FilterCatalog(params)

print("=" * 75)
print("  PAINS Filter Analysis")
print("=" * 75)
print("\nPAINS = Pan-Assay Interference Compounds")
print("These are compounds that give false positives in many assays.\n")

# ─── Check each ligand ───
pains_data = {}

for lid, mol in ligands.items():
    matches = pains_catalog.GetMatches(mol)
    n_alerts = len(matches)
    
    alerts_list = []
    for match in matches:
        alerts_list.append(match.GetDescription())
    
    pains_data[lid] = {
        "n_alerts": n_alerts,
        "alerts": alerts_list,
        "verdict": "Clean" if n_alerts == 0 else "Flagged",
    }
    
    print(f"\n▶ {lid}:")
    if n_alerts == 0:
        print(f"   ✅ Clean — No PAINS alerts")
    else:
        print(f"   ⚠️  {n_alerts} PAINS alert(s):")
        for alert in alerts_list[:5]:  # Show first 5
            print(f"      - {alert}")

# ─── Summary Table ───
print("\n" + "=" * 75)
print("  PAINS Summary Table")
print("=" * 75)

summary = {lid: data["n_alerts"] for lid, data in pains_data.items()}
df_pains = pd.DataFrame.from_dict(summary, orient="index", columns=["PAINS_Alerts"])
df_pains["Verdict"] = df_pains["PAINS_Alerts"].apply(
    lambda x: "✅ Clean" if x == 0 else "⚠️ Flagged"
)
print(df_pains.to_string())
print("=" * 75)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 8: Radar Chart — Visual Comparison
# ═══════════════════════════════════════════════════════════

# ─── Select descriptors to plot ───
radar_props = ["MW", "LogP", "HBD", "HBA", "TPSA", "RotB"]

# ─── Normalize values (0-1 scale) ───
df_radar = df_descriptors[radar_props].copy()

# Max values for normalization (typical upper limits)
max_vals = {
    "MW": 600, "LogP": 6, "HBD": 6,
    "HBA": 12, "TPSA": 200, "RotB": 15,
}

for prop in radar_props:
    df_radar[prop] = df_radar[prop] / max_vals[prop]

# ─── Setup angles ───
n_props = len(radar_props)
angles = np.linspace(0, 2 * np.pi, n_props, endpoint=False).tolist()
angles += angles[:1]  # Close the polygon

# ─── Create the plot ───
fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

# Color palette
colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12"]

for (lid, row), color in zip(df_radar.iterrows(), colors):
    values = row.tolist()
    values += values[:1]  # Close the polygon
    ax.plot(angles, values, "o-", linewidth=2, label=lid, color=color)
    ax.fill(angles, values, alpha=0.15, color=color)

# ─── Customize ───
ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_props, fontsize=12, fontweight="bold")
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["25%", "50%", "75%", "100%"], fontsize=9)
ax.set_title(
    "ADMET Radar Chart — Top 4 Ligands\n(Normalized to typical upper limits)",
    fontsize=14, fontweight="bold", pad=20,
)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=11)

plt.tight_layout()
plt.savefig("docking/results/admet_radar.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n✅ Saved: docking/results/admet_radar.png")

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 9: Bar Chart — Per-Property Comparison
# ═══════════════════════════════════════════════════════════

# ─── Create 2x3 grid ───
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

# ─── Properties to plot ───
props_to_plot = [
    ("MW",   "Molecular Weight (Da)",         500, "Lipinski limit"),
    ("LogP", "LogP (lipophilicity)",            5, "Lipinski limit"),
    ("HBD",  "H-Bond Donors",                   5, "Lipinski limit"),
    ("HBA",  "H-Bond Acceptors",               10, "Lipinski limit"),
    ("TPSA", "TPSA (Å²)",                      140, "Veber limit"),
    ("RotB", "Rotatable Bonds",                10, "Veber limit"),
]

# ─── Colors ───
ligand_colors = {
    "607": "#e74c3c",
    "BAH": "#3498db",
    "13U": "#2ecc71",
    "12U": "#f39c12",
}

ligand_ids = list(ligands.keys())

# ─── Plot each property ───
for i, (prop, ylabel, limit, limit_label) in enumerate(props_to_plot):
    ax = axes[i]
    
    values = [descriptors_data[lid][prop] for lid in ligand_ids]
    colors = [ligand_colors[lid] for lid in ligand_ids]
    
    bars = ax.bar(ligand_ids, values, color=colors, edgecolor="black", linewidth=1.5)
    
    # Add threshold line
    ax.axhline(y=limit, color="red", linestyle="--", linewidth=2, 
               label=limit_label)
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, height + (max(values) * 0.03),
                f"{val:.2f}" if isinstance(val, float) else str(val),
                ha="center", fontsize=10, fontweight="bold")
    
    ax.set_ylabel(ylabel, fontsize=11, fontweight="bold")
    ax.set_title(f"{prop}", fontsize=13, fontweight="bold")
    ax.legend(fontsize=9, loc="upper right")
    ax.grid(axis="y", alpha=0.3)
    ax.set_ylim(0, max(values + [limit]) * 1.25)

plt.suptitle("ADMET Properties — Top 4 Ligands", 
             fontsize=16, fontweight="bold", y=1.00)
plt.tight_layout()
plt.savefig("docking/results/admet_barchart.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n✅ Saved: docking/results/admet_barchart.png")

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 10: Save All Results to Files
# ═══════════════════════════════════════════════════════════

# ─── Create output directory ───
ADMET_DIR = "docking/results/admet"
os.makedirs(ADMET_DIR, exist_ok=True)

print("=" * 70)
print("  Saving ADMET Results")
print("=" * 70)
print()

# ─── 1. Save Descriptors ───
path = f"{ADMET_DIR}/admet_descriptors.csv"
df_descriptors.to_csv(path)
print(f"✅ {path}")
print(f"   → {df_descriptors.shape[0]} ligands × {df_descriptors.shape[1]} descriptors")

# ─── 2. Save Lipinski Results ───
path = f"{ADMET_DIR}/admet_lipinski.csv"
df_lipinski.to_csv(path)
print(f"✅ {path}")

# ─── 3. Save Veber Results ───
path = f"{ADMET_DIR}/admet_veber.csv"
df_veber.to_csv(path)
print(f"✅ {path}")

# ─── 4. Save PAINS Results ───
path = f"{ADMET_DIR}/admet_pains.csv"
df_pains.to_csv(path)
print(f"✅ {path}")

# ─── 5. Build combined summary ───
summary_data = []

for lid in ligand_ids:
    row = {
        "Ligand": lid,
        "MW": descriptors_data[lid]["MW"],
        "LogP": descriptors_data[lid]["LogP"],
        "HBD": descriptors_data[lid]["HBD"],
        "HBA": descriptors_data[lid]["HBA"],
        "TPSA": descriptors_data[lid]["TPSA"],
        "RotB": descriptors_data[lid]["RotB"],
        "Lipinski_Violations": lipinski_data[lid]["violations"],
        "Lipinski_Verdict": lipinski_data[lid]["verdict"],
        "Veber_Verdict": veber_data[lid]["verdict"],
        "PAINS_Alerts": pains_data[lid]["n_alerts"],
    }
    summary_data.append(row)

df_final = pd.DataFrame(summary_data)
df_final = df_final.set_index("Ligand")

path = f"{ADMET_DIR}/admet_summary.csv"
df_final.to_csv(path)
print(f"✅ {path}")

print()
print("=" * 70)
print(f"  🎉 All {5} files saved to: {ADMET_DIR}/")
print("=" * 70)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Cell 11: Final Summary & Scientific Verdict
# ═══════════════════════════════════════════════════════════

print("=" * 100)
print("  FINAL ADMET SUMMARY — Top 4 Ligands")
print("=" * 100)

# ─── Display the full summary table ───
print("\n📊 Complete Property Table:")
print("-" * 100)
print(df_final.to_string())
print("-" * 100)

# ─── Verdict per ligand ───
print("\n🔬 Scientific Verdict per Ligand:")
print("=" * 100)

for lid in ligand_ids:
    lip = lipinski_data[lid]
    veb = veber_data[lid]
    pns = pains_data[lid]
    props = descriptors_data[lid]
    
    # Compute overall verdict
    is_druglike = (
        lip["violations"] <= 1 and
        veb["verdict"] == "Pass" and
        pns["n_alerts"] == 0
    )
    
    verdict_icon = "✅" if is_druglike else "⚠️"
    verdict_text = "DRUG-LIKE" if is_druglike else "Needs Review"
    
    print(f"\n▶ {lid} {verdict_icon} — {verdict_text}")
    print(f"   MW:     {props['MW']:.2f} Da")
    print(f"   LogP:   {props['LogP']:.2f}")
    print(f"   HBD:    {props['HBD']}")
    print(f"   HBA:    {props['HBA']}")
    print(f"   TPSA:   {props['TPSA']:.2f} Å²")
    print(f"   RotB:   {props['RotB']}")
    print(f"   ─────────────────────────────────")
    print(f"   Lipinski:  {lip['violations']} violations → {lip['verdict']}")
    print(f"   Veber:     {veb['verdict']}")
    print(f"   PAINS:     {pns['n_alerts']} alerts → {pns['verdict']}")

print("\n" + "=" * 100)
print("  🎉 ADMET Analysis Complete!")
print("=" * 100)

---

## 🎉 Lesson 15 — Summary

### ✅ What We Learned

**1. Molecular Descriptors (RDKit)**
- Computed 12 properties for each ligand
- MW, LogP, HBD, HBA, TPSA, RotB, and more

**2. Drug-Likeness Filters (3 Layers)**
- **Lipinski's Rule of 5** — Oral absorption
- **Veber's Rules** — Oral bioavailability
- **PAINS Filter** — Pan-assay interference

**3. Visualizations**
- **Radar Chart** — Holistic comparison
- **Bar Chart** — Per-property comparison

---

### 📊 Final Verdict

Each ligand was evaluated against **3 filter layers**:

| Filter | Criterion | Pass = ? |
|--------|-----------|----------|
| **Lipinski** | ≤ 1 violation | Drug-like |
| **Veber** | RotB ≤ 10 AND TPSA ≤ 140 | Pass |
| **PAINS** | 0 alerts | Clean |

**Final Verdicts:**
- **607** → Depends on results
- **BAH** → Depends on results
- **13U** → Depends on results
- **12U** → Depends on results

*(Fill in the actual verdicts after running the cells)*

---

### 🚀 What This Means

**Drug-like ligands** → Move to MD Simulations

**Non-drug-like ligands** → Document and either:
- Optimize structure (medicinal chemistry)
- Exclude from further study

---

### 📁 Files Produced

docking/results/admet/

    admet_descriptors.csv ← All 12 properties

├── admet_lipinski.csv ← Lipinski results

├── admet_veber.csv ← Veber results

├── admet_pains.csv ← PAINS alerts

├── admet_summary.csv ← Final combined table

├── admet_radar.png ← Radar chart

└── admet_barchart.png ← Bar charts


---

### 🎯 Connection to Lesson 14b

**Lesson 14b** gave us: **Which ligands bind?** (Docking + ProLIF)
**Lesson 15** gives us: **Can they be drugs?** (ADMET)

**Together:** A complete drug-discovery pipeline!

---

### 📌 Next Steps

1. **Move to MD Simulations** (for drug-like ligands)
2. **Chemical Optimization** (for non-drug-like)
3. **Experimental Validation** (if resources allow)

---

**End of Lesson 15** 🎉